# Cinco hilos CUDA contando del 1 al 10

Este ejemplo está pensado para Linux con una GPU NVIDIA. Lanza un bloque de exactamente 5 hilos CUDA; cada hilo escribe los números del 1 al 10 en la GPU. Después Python copia e imprime los resultados.

Ejecuta la celda de instalación y reinicia el kernel antes de ejecutar el ejemplo. Numba necesita también las bibliotecas del CUDA Toolkit para compilar el kernel.

In [ ]:
%pip install -U 'numba-cuda[cu12]'

In [ ]:
import numpy as np
from numba import cuda


@cuda.jit
def contar_en_gpu(resultados):
    hilo = cuda.grid(1)
    if hilo < 5:
        for numero in range(1, 11):
            resultados[hilo, numero - 1] = numero


if not cuda.is_available():
    import torch
    raise RuntimeError(
        f"Numba no puede usar CUDA en este kernel. "
        f"PyTorch detecta CUDA: {torch.cuda.is_available()}. "
        f"Ejecuta la celda de instalación de numba-cuda[cu12] y reinicia el kernel."
    )

resultados_gpu = cuda.device_array((5, 10), dtype=np.int32)
contar_en_gpu[1, 5](resultados_gpu)  # Un bloque con cinco hilos CUDA
resultados = resultados_gpu.copy_to_host()

for hilo, fila in enumerate(resultados, start=1):
    for numero in fila:
        print(f"Hilo {hilo}: {numero}")